In [13]:
from google.colab import files
uploaded = files.upload()

In [14]:
#check dataset
from datasets import load_dataset

train_path = "/content/train.jsonl"
val_path = "/content/val.jsonl"

train_dataset = load_dataset("json", data_files=train_path, split="train")
val_dataset = load_dataset("json", data_files=val_path, split="train")
# In thử ra vài mẫu kiểm tra
print(train_dataset[1])
print(val_dataset[0])

{'text': '<|user|>\nCreate reading passage for IELTS on public health</s>\n<|assistant|>\nReading Passage:\n\nIn recent years, public health has become an increasingly important topic in the field of health. Researchers and experts have devoted considerable attention to understanding its implications and potential impact on society. One of the primary considerations regarding public health involves its practical applications. Studies have shown that implementing effective strategies can lead to significant improvements in outcomes. Researchers have documented numerous cases where innovative approaches have yielded positive results. Furthermore, the integration of modern methodologies has enhanced our ability to address related challenges. These developments suggest that continued investment in this area will likely produce substantial benefits for society as a whole. The historical context of public health provides valuable insights into current trends. Early pioneers in health laid th

In [15]:
#load model
from transformers import pipeline
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
pipe = pipeline(task='text-generation', model=model_name, device='cuda')

Device set to use cuda


In [16]:
!pip install -q accelerate -U
!pip install -q bitsandbytes -U
!pip install -q trl -U
!pip install -q peft -U
!pip install -q transformers -U
!pip install -q datasets -U

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 362, in run
    resolver = self.make_resolver(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 177, in make_resolver
    return pip._internal.resolution.resolvelib.resolver.Resolver(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 58, in __init__
    self.factory = Factory(
                   ^^^^^^^^
  File "/usr/local/lib/py

In [17]:
import torch
print(torch.cuda.is_available())

True


In [18]:
print(torch.__version__)

2.8.0+cu126


In [19]:
#configuring Qlora
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype='float16',
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)
model.config.use_cache = False
model.config.pretraining_tp = 1

In [20]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=64,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [21]:
from transformers import TrainingArguments
from trl import SFTTrainer
output_dir = "train_dir"

args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=3,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy ="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=args,
    peft_config=peft_config
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [22]:
import os
os.environ["WANDB_DISABLED"] = "true"
import wandb
wandb.init(mode="disabled")

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,0.746000,0.311987,0.445978,81917.000000,0.927332
20,0.172200,0.083414,0.117113,163807.000000,0.977216
30,0.065800,0.053439,0.068562,245713.000000,0.982491
40,0.049000,0.046648,0.055698,327605.000000,0.982254
50,0.043900,0.044014,0.048210,409507.000000,0.983790
60,0.042700,0.043023,0.047060,491423.000000,0.983617
70,0.041800,0.042111,0.044878,573343.000000,0.983916
80,0.041100,0.041159,0.044264,655230.000000,0.984061
90,0.042000,0.040850,0.043703,737148.000000,0.983903
100,0.042100,0.040351,0.043709,818970.000000,0.984037


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant p

TrainOutput(global_step=1824, training_loss=0.041560718985764605, metrics={'train_runtime': 6980.4, 'train_samples_per_second': 2.089, 'train_steps_per_second': 0.261, 'total_flos': 9.719117145693389e+16, 'train_loss': 0.041560718985764605, 'epoch': 3.0})

In [23]:
!zip -r final_model.zip "/content/train_dir/checkpoint-1824"
from google.colab import files
files.download("final_model.zip")

  adding: content/train_dir/checkpoint-1824/ (stored 0%)
  adding: content/train_dir/checkpoint-1824/chat_template.jinja (deflated 60%)
  adding: content/train_dir/checkpoint-1824/README.md (deflated 65%)
  adding: content/train_dir/checkpoint-1824/tokenizer.model (deflated 55%)
  adding: content/train_dir/checkpoint-1824/rng_state.pth (deflated 26%)
  adding: content/train_dir/checkpoint-1824/trainer_state.json (deflated 82%)
  adding: content/train_dir/checkpoint-1824/training_args.bin (deflated 53%)
  adding: content/train_dir/checkpoint-1824/adapter_config.json (deflated 57%)
  adding: content/train_dir/checkpoint-1824/scaler.pt (deflated 64%)
  adding: content/train_dir/checkpoint-1824/special_tokens_map.json (deflated 72%)
  adding: content/train_dir/checkpoint-1824/optimizer.pt (deflated 9%)
  adding: content/train_dir/checkpoint-1824/adapter_model.safetensors (deflated 8%)
  adding: content/train_dir/checkpoint-1824/scheduler.pt (deflated 62%)
  adding: content/train_dir/checkp

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>